# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a simple, efficient Convolutional Neural Network (CNN) with residual connections on the MNIST dataset using PyTorch Lightning. The configuration is tuned for ≥99.7% accuracy.

In [ ]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn tsilva_notebook_utils==0.0.67

In [ ]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

Define config:

In [ ]:
import os

def setup_config():
    # General Settings
    model_id = "resnet18"#"resnet50"
    backbone_warmup_percentage = 0.1
    dataset_id = "cifar10"  # Options: "mnist" or "cifar10"
    pretrained_dataset_id = "imagenet"
    seed = 42
    n_epochs = 100
    batch_size = 256
    # Optimizer Settings
    learning_rate = 1e-3
    weight_decay = 0
    # Model Architecture
    nonlinearity = 'relu'
    # Data Settings
    train_size = 0.8
    val_size = 0.2
    # Softcoded new params
    label_smoothing = 0.0
    early_stopping_patience = 7
    early_stopping_min_delta = 1e-4
    precision = "16-mixed"
    swa_lrs = 0#1e-2
    activation = "relu"
    
    augmentation_pipeline = [
        ("RandomCrop", [], dict(size=32, padding=4)),
        ("RandomHorizontalFlip", [], dict(p=0.5)),
        ("ColorJitter", [], dict(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)),
        ("RandomErasing", [], dict(p=0.25, scale=(0.02, 0.33), ratio=(0.3, 3.3))),
    ]

    # TODO: softcode this
    os.environ["NOTEBOOK_ID"] = "mnist-cnn-pl"

    return {
        'model_id': model_id,
        "pretrained_dataset_id": pretrained_dataset_id,
        'backbone_warmup_percentage' : backbone_warmup_percentage,
        'dataset_id': dataset_id,
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'nonlinearity': nonlinearity,
        'weight_decay': weight_decay,
        'train_size': train_size,
        'val_size': val_size,
        'label_smoothing': label_smoothing,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_min_delta': early_stopping_min_delta,
        'precision': precision,
        'swa_lrs': swa_lrs,
        'activation': activation,
        'augmentation_pipeline': augmentation_pipeline
    }

CONFIG = setup_config()

Set seed for reproducibility:

In [ ]:
import pytorch_lightning as pl
pl.seed_everything(CONFIG['seed'], workers=True)

Set matmul precision:

In [ ]:
import torch
torch.set_float32_matmul_precision('high')

Login to wandb:

In [ ]:
import wandb
wandb.login()

Create the data module:

In [ ]:
from tsilva_notebook_utils.lightning import create_data_module
dm = create_data_module(CONFIG)
dm

Create the model:

In [ ]:
from torch import nn

class LitModel(pl.LightningModule):
    def __init__(self, lr=None, n_classes=None):
        super().__init__()

        self.save_hyperparameters()

        assert n_classes is not None, "n_classes must be provided"
        
        # Load model from torch hub
        self.model = torch.hub.load("pytorch/vision", CONFIG["model_id"], weights="DEFAULT")

        # Swap out classification head (eg: pretrained weights may have different number of classes)
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

        self.criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
    
    def freeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = False
        for param in self.model.fc.parameters(): param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters(): param.requires_grad = True
        for param in self.model.fc.parameters(): param.requires_grad = True

    def forward(self, x):
        x = self.model(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train/loss", loss, prog_bar=True)
        self.log("train/acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val/loss", loss, prog_bar=True, sync_dist=True)
        self.log("val/acc", acc, prog_bar=True, sync_dist=True)
        return {}

    def on_validation_epoch_end(self):
        pass

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test/loss", loss)
        self.log("test/acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=CONFIG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val/loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
model

Find optimal batch size:

In [ ]:
#from pytorch_lightning.tuner import Tuner
#trainer = pl.Trainer()
#tuner = Tuner(trainer)
#tuner.scale_batch_size(model, datamodule=dm, mode="power")

First try to overfit a batch to make sure the training pipeline works:

In [ ]:
from tsilva_notebook_utils.lightning import ThresholdStoppingCallback

trainer = pl.Trainer(
    log_every_n_steps=1,
    overfit_batches=1,
    max_epochs=100,
    callbacks=[ThresholdStoppingCallback("train/acc", 1.0)]
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=create_data_module(CONFIG, batch_size=32, train_shuffle=False))

Run the training loop:

In [ ]:
from tsilva_notebook_utils.lightning import EpochTimeLogger, BackboneWarmupCallback
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import StochasticWeightAveraging

num_gpus = torch.cuda.device_count()
trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",
    benchmark=True,
    max_epochs=CONFIG['n_epochs'],
    log_every_n_steps=5,
    logger=WandbLogger(project=os.environ["NOTEBOOK_ID"], config=CONFIG),
    precision=CONFIG['precision'],
    callbacks=[x for x in [
        EpochTimeLogger(), 
        LearningRateMonitor(logging_interval="epoch"),
        EarlyStopping(
            monitor="val/loss",
            patience=CONFIG['early_stopping_patience'],
            min_delta=CONFIG['early_stopping_min_delta'],
            mode="min",
            verbose=True,
            strict=True
        ) if CONFIG['early_stopping_patience'] > 0 else None,
        StochasticWeightAveraging(swa_lrs=CONFIG['swa_lrs']) if CONFIG['swa_lrs'] > 0 else None,
        BackboneWarmupCallback(CONFIG['backbone_warmup_percentage']) if CONFIG['backbone_warmup_percentage'] > 0 else None
    ] if x is not None]
)
model = LitModel(
    lr=CONFIG['learning_rate'],
    n_classes=dm.n_classes
)
trainer.fit(model, datamodule=dm)


Calculate the post-training test set accuracy:

In [ ]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)